In [1]:
import fatesens as fs
import scanpy as sc
import numpy as np
import pandas as pd

/Users/ps-lab-noman/Research/FateSens/.venv/lib/python3.12/site-packages/tqdm_joblib/__init__.py:4: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [2]:
day_t0 = [2, 4]
day_t1 = [6]
day_column_name = "time_info"
n_neighbor_for_jacobian = 200

In [3]:
adata = sc.read(
    "../data/adata.h5ad"
)
# adata = sc.pp.subsample(adata, fraction=0.2, copy=True)
adata

AnnData object with n_obs × n_vars = 34782 × 23420
    obs: 'Library', 'Cell barcode', 'time_info', 'Starting population', 'state_info', 'Well', 'SPRING-x', 'SPRING-y', 'Neutrophil_pDC_raw_score', 'Neutrophil_pDC_smoothed_score', 'Neutrophil_pDC_fate_class', 'Neutrophil_Monocyte_raw_score', 'Neutrophil_Monocyte_smoothed_score', 'Neutrophil_Monocyte_fate_class', 'is_boundary_between_clonal_sisters_Neutrophil_Monocyte', 'is_boundary_between_clonal_sisters_Neutrophil_pDC'
    var: 'Accession', 'Chromosome', 'End', 'Start', 'Strand'
    uns: 'data_des', 'neighbors', 'state_info_colors'
    obsm: 'X_clone', 'X_emb'
    layers: 'ambiguous', 'matrix', 'spliced', 'unspliced'
    obsp: 'connectivities', 'distances'

### Preprocessing

In [4]:
adata = fs.pp.get_highly_variable_genes_subset(adata)

### Flow map estimation

In [5]:
x_0 = fs.flow_map.get_day_t0_expression(adata, days_t0=day_t0,)
x_t_probability = fs.flow_map.calculate_fate_probability(adata, day_t0=day_t0,day_t1=day_t1,)

In [6]:
jacobians = fs.jacobian_matrix.estimate_jacobian_of_fate_probability(adata, x_0, x_t_probability, days_t0=day_t0, day_column_name=day_column_name,n_neighbors=20,beta=10,)

Function: get_jacobian, Workers: 5
Running in parallel with 5/10 jobs


Processing:   0%|          | 0/14508 [00:00<?, ?it/s]

### Fate boundary

In [7]:
ftle = fs.jacobian_matrix.compute_largest_singular_values(jacobians)

Function: get_largest_singular_value, Workers: 5
Running in parallel with 5/10 jobs


Processing:   0%|          | 0/14508 [00:00<?, ?it/s]

In [ ]:
ridge_adata_obs = fs.tl.estimate_ridge(adata, sensitivities=ftle, radius=100)

The current bandwidth is 171.49130035793095.

